# #1
Deep Networks: VGGNet은 신경망의 깊이가 모델의 성능에 미치는 영향을 집중적으로 연구했습니다. 층을 16~19층까지 깊게 쌓을수록 이미지 분류 성능이 획기적으로 향상된다는 것을 입증했습니다.  

Small Filters & Max Pooling: 이전 모델들이 사용하던 큰 필터(11x11 등)를 과감히 버리고, 아주 작은 3x3 필터만 연속으로 사용했습니다. 공간의 크기를 줄이고 중요한 특징을 압축하기 위해서는 2x2 크기의 Max Pooling을 사용했습니다.  

Increased Non-Linearity & ReLU Activation Function: 큰 필터 1개를 쓰는 것보다 3x3 필터를 여러 번 겹쳐 쓰면, 각 층을 통과할 때마다 ReLU 활성화 함수를 더 많이 거치게 됩니다. 파라미터(가중치) 개수는 오히려 줄어들면서도 데이터의 복잡한 특징을 더 잘 구분해 내는 비선형성은 크게 증가하는 효과를 얻었습니다.

Simple Architecture: 3x3 합성곱 층과 2x2 풀링 층만을 반복해서 쌓아 올린 매우 직관적이고 단순한 구조를 채택했습니다.   

ILSVRC Performance: 이 단순하고 깊은 아키텍처를 바탕으로 2014년 이미지넷 대회에서 훌륭한 성적을 거두었으며, 딥러닝 발전 역사에 있어 모델을 깊게 쌓는 방법론의 강력한 기준이 되었습니다.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. 데이터 전처리 및 데이터 로더 설정
transform = transforms.Compose([
    transforms.Resize(72),
    transforms.RandomCrop(56),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = datasets.STL10(root='./data', split='train', download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)


# 2. VGGNet 모델 클래스 구성
class VGGNet11(nn.Module):
    def __init__(self, num_classes=10):
        super(VGGNet11, self).__init__()

        self.convnet = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.fclayer = nn.Sequential(
            nn.Linear(512, 4096),
            nn.ReLU(),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.convnet(x)
        x = torch.flatten(x, 1)
        output = self.fclayer(x)
        return output


# 3. 모델, 손실 함수, 옵티마이저 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VGGNet11(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)


# 4. 모델 학습
epochs = 10

for epoch in range(1, epochs + 1):
    model.train()

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        if batch_idx % 10 == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} '
                  f'({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}')

100%|██████████| 2.64G/2.64G [06:08<00:00, 7.17MB/s]


Train Epoch: 1 [0/5000 (0%)]	Loss: 2.301893
Train Epoch: 1 [2560/5000 (50%)]	Loss: 2.304252
Train Epoch: 2 [0/5000 (0%)]	Loss: 2.302593
Train Epoch: 2 [2560/5000 (50%)]	Loss: 2.302939
Train Epoch: 3 [0/5000 (0%)]	Loss: 2.302780
Train Epoch: 3 [2560/5000 (50%)]	Loss: 2.302330
Train Epoch: 4 [0/5000 (0%)]	Loss: 2.301427
Train Epoch: 4 [2560/5000 (50%)]	Loss: 2.297577
Train Epoch: 5 [0/5000 (0%)]	Loss: 2.280372
Train Epoch: 5 [2560/5000 (50%)]	Loss: 2.222645
Train Epoch: 6 [0/5000 (0%)]	Loss: 2.133824
Train Epoch: 6 [2560/5000 (50%)]	Loss: 2.081958
Train Epoch: 7 [0/5000 (0%)]	Loss: 2.099302
Train Epoch: 7 [2560/5000 (50%)]	Loss: 2.057766
Train Epoch: 8 [0/5000 (0%)]	Loss: 2.075117
Train Epoch: 8 [2560/5000 (50%)]	Loss: 2.028888
Train Epoch: 9 [0/5000 (0%)]	Loss: 2.042848
Train Epoch: 9 [2560/5000 (50%)]	Loss: 2.035683
Train Epoch: 10 [0/5000 (0%)]	Loss: 1.970954
Train Epoch: 10 [2560/5000 (50%)]	Loss: 1.969848


# #2

Deep Networks : 신경망은 층을 깊게 쌓을수록 성능이 향상될 것으로 기대되지만, 실제로는 일정 깊이 이상에서 오히려 학습이 안 되고 오차가 커지는 성능 저하 현상이 발생합니다. ResNet은 이 문제를 극복하고 모델을 극단적으로 깊게 쌓기 위해 고안되었습니다.

Residual Learning : 기존 신경망이 입력값 x를 타겟값 H(x)로 직접 변환하도록 학습했다면, ResNet은 출력과 입력의 차이인 잔차 F(x) = H(x) - x$를 학습하도록 목표를 변경했습니다. 즉, 모델이 0에 가까운 미세한 변화만 학습하도록 하여 최적화 난이도를 대폭 낮추었습니다.

Skip Connection : 잔차 학습을 구현하기 위해 입력값 x를 합성곱 층을 거치지 않고 바로 출력단에 더해주는 일종의 지름길 구조를 도입했습니다. 이 지름길 덕분에 역전파 시 모델의 앞쪽 층까지 기울기가 잘 전달되어 기울기 소실 문제를 해결하고, 152층 이상의 매우 깊은 네트워크에서도 성능을 크게 개선했습니다

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import resnet50
from torch.utils.data import DataLoader

# 1. 데이터 전처리 및 로더 설정
transform = transforms.Compose([
    transforms.Resize(72),
    transforms.RandomCrop(56),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = datasets.STL10(root='./data', split='train', download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)


# 2. ResNet 모델 구성
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = resnet50(weights=None)

model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)


# 3. 손실 함수 및 옵티마이저 설정
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


# 4. 모델 직접 학습시키는 로직
epochs = 10

for epoch in range(1, epochs + 1):
    model.train()

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        if batch_idx % 10 == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} '
                  f'({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}')

Train Epoch: 1 [0/5000 (0%)]	Loss: 2.452264
Train Epoch: 1 [2560/5000 (50%)]	Loss: 2.241472
Train Epoch: 2 [0/5000 (0%)]	Loss: 2.035698
Train Epoch: 2 [2560/5000 (50%)]	Loss: 1.803031
Train Epoch: 3 [0/5000 (0%)]	Loss: 1.668567
Train Epoch: 3 [2560/5000 (50%)]	Loss: 1.562074
Train Epoch: 4 [0/5000 (0%)]	Loss: 1.527134
Train Epoch: 4 [2560/5000 (50%)]	Loss: 1.490504
Train Epoch: 5 [0/5000 (0%)]	Loss: 1.613063
Train Epoch: 5 [2560/5000 (50%)]	Loss: 1.556898
Train Epoch: 6 [0/5000 (0%)]	Loss: 1.375852
Train Epoch: 6 [2560/5000 (50%)]	Loss: 1.364653
Train Epoch: 7 [0/5000 (0%)]	Loss: 1.318888
Train Epoch: 7 [2560/5000 (50%)]	Loss: 1.345830
Train Epoch: 8 [0/5000 (0%)]	Loss: 1.544526
Train Epoch: 8 [2560/5000 (50%)]	Loss: 1.431051
Train Epoch: 9 [0/5000 (0%)]	Loss: 1.444372
Train Epoch: 9 [2560/5000 (50%)]	Loss: 1.450077
Train Epoch: 10 [0/5000 (0%)]	Loss: 1.481439
Train Epoch: 10 [2560/5000 (50%)]	Loss: 1.241425
